# Assignment 1: EDA ve Baseline
**Yarışma:** Deep Past Initiative – Machine Translation (Akadca → İngilizce)

Bu notebook'un akışı: Kurulum → Veri → EDA → Baseline modelleri → Sonuç tablosu.
Tekrar kullanılan kodlar `src/` klasöründedir (`src/data.py`, `src/models.py`).

## 1. Kurulum

In [ ]:
import os, subprocess

REPO = "GROUP36_CENG313"
URL = "https://github.com/MetehanBB/GROUP36_CENG313.git"

# Zaten repo klasöründeysek tekrar klonlama
if os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-b", "metehan", URL], check=True)
    os.chdir(REPO)

subprocess.run(["pip", "install", "-q", "sacrebleu"], check=True)
print("Çalışma dizini:", os.getcwd())

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sys.path.append(".")
from src.data import clean_akkadian_text
from src.models import split_data, run_baselines

sns.set_theme(style="whitegrid")
print("Kütüphaneler ve src modülleri yüklendi.")

## 2. Veri

In [ ]:
DATA_DIR = "deep-past-initiative-machine-translation"
print("Veri klasöründeki dosyalar:", os.listdir(DATA_DIR))

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(f"\nTrain boyutu (belge düzeyi): {train_df.shape}")
print(f"Test boyutu (cümle düzeyi): {test_df.shape}")
print(f"Sample submission boyutu: {sample_sub.shape}")

In [ ]:
print("--- Train eksik değerler ---")
print(train_df.isnull().sum())
print("\n--- Test eksik değerler ---")
print(test_df.isnull().sum())

In [ ]:
train_df.head(3)

**Yorum:** _Buraya 1-2 cümle yaz: kaç satır var, eksik değer var mı, train ve test neden farklı düzeyde (belge / cümle)?_

## 3. Keşifsel Veri Analizi (EDA)

### 3.1 Metin uzunlukları

In [ ]:
train_df["translit_len"] = train_df["transliteration"].astype(str).apply(len)
train_df["translation_len"] = train_df["translation"].astype(str).apply(len)
train_df[["translit_len", "translation_len"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(train_df["translit_len"], bins=30, color="skyblue", edgecolor="black")
axes[0].set_title("Akkadian Transliteration Lengths")
axes[0].set_xlabel("Karakter sayısı")
axes[0].set_ylabel("Örnek sayısı")

axes[1].hist(train_df["translation_len"], bins=30, color="salmon", edgecolor="black")
axes[1].set_title("English Translation Lengths")
axes[1].set_xlabel("Karakter sayısı")
axes[1].set_ylabel("Örnek sayısı")

plt.tight_layout()
plt.show()

**Yorum:** _Buraya 1-2 cümle yaz: dağılımlar nasıl görünüyor (tek tepeli mi, iki tepeli mi, uzun kuyruk var mı)? Akadca ve İngilizce uzunlukları birbirine benziyor mu?_

### 3.2 En sık geçen kelimeler (transliterasyon)

In [ ]:
tokens = Counter(" ".join(train_df["transliteration"].dropna().astype(str)).split())
top = pd.DataFrame(tokens.most_common(15), columns=["kelime", "sayı"])

plt.figure(figsize=(8, 5))
sns.barplot(data=top, y="kelime", x="sayı", color="skyblue")
plt.title("En sık 15 token")
plt.tight_layout()
plt.show()
print("Farklı token sayısı:", len(tokens))

**Yorum:** _Buraya 1-2 cümle yaz: en sık tokenlar ne tür (sayılar, işaretler, kelimeler)? Kelime dağarcığı büyük mü?_

### 3.3 Tekrar eden ve temizlenen metinler

In [ ]:
dup_src = train_df["transliteration"].duplicated().sum()
print(f"Tekrar eden transliterasyon sayısı: {dup_src}")

# Metin temizleme fonksiyonu (src/data.py) örnek üzerinde
sample_text = train_df["transliteration"].iloc[0]
print("\nOrijinal :", sample_text[:150])
print("Temizlenmiş:", clean_akkadian_text(sample_text)[:150])

**Yorum:** _Buraya 1-2 cümle yaz: tekrar eden satır var mı, bu baseline sonuçlarını nasıl etkileyebilir?_

## 4. Baseline modelleri

Üç basit baseline, aynı train/validation bölmesiyle (%80 / %20, `random_state=42`) karşılaştırılır:

1. **Copy-input:** Akadca girdi aynen çıktı olarak verilir.
2. **Sabit cümle:** Train'deki en sık çeviri her örnek için verilir.
3. **TF-IDF en yakın komşu:** Train'deki en benzer girdinin çevirisi verilir.

Kodlar `src/models.py` içindedir. Metrikler: BLEU ve chrF (`sacrebleu`).

In [ ]:
train_part, val_part = split_data(train_df)
print(f"Train: {len(train_part)} satır | Validation: {len(val_part)} satır")

results = run_baselines(train_part, val_part)
results

**Yorum:** TF-IDF en yakın komşu yöntemi hem BLEU hem chrF bakımından açık farkla en iyi baseline oldu. Tabletlerde mühür ve "X oğlu Y" gibi kalıplar sık tekrar ettiği için, train'deki en benzer cümlenin çevirisini vermek işe yarıyor. Copy-input en düşük skoru aldı, çünkü Akadca transliterasyon ile İngilizce çeviri arasında neredeyse ortak kelime yok. Sabit cümle ise İngilizce çıktı verdiği için karakter düzeyinde (chrF) biraz puan kazandı. Skorlar rastgele train/validation bölmesiyle hesaplandı. Aynı belgeden gelen parçalar iki tarafa da düşmüş olabileceği için gerçek test skorunun üzerinde olabilir. Sonraki adımda geliştirilecek modellerin bu üç değeri geçmesi hedeflenmektedir.